# 0. Data Acquisition, Sampling & 2-Core Filtering

## Purpose
This notebook builds the foundational dataset for our book recommendation and cold-start pipeline using the Stanford SNAP Amazon Books dataset (`reviews_Books_5.json.gz` and `meta_Books.json.gz`). To handle the large file sizes (~4.0 GB compressed) within local RAM limits, it streams the raw files in chunks, performs deterministic user sampling, applies iterative $k$-core filtering for graph stability, and extracts matching item metadata.

## Pipeline Overview
1. **Safe Chunked Download:** Downloads raw SNAP review and metadata archives with GZIP integrity verification.
2. **Deterministic User Sampling (50%):** Uses CRC32 hashing (`seed=42`) on `user_id` to retain 50% of users (4,424,388 reviews across 365,224 items).
3. **Iterative 2-Core Filtering:** Filters out users and items with fewer than 2 interactions until convergence, ensuring every node in downstream co-review graphs has valid structural connectivity.
4. **Metadata Extraction:** Streams `meta_Books.json.gz` and extracts structured attributes (`title`, `author`, `description`, `features`, `categories`, `publisher`, `page_count`, etc.) for all surviving items.

## Inputs & Outputs
* **Inputs (Auto-downloaded to `data/`):**
  * `data/reviews_Books_5.json.gz` (3.22 GB)
  * `data/meta_Books.json.gz` (826 MB)
* **Outputs (Cached to `data/`):**
  * `data/books_50pct_users_reviews.parquet` — 4,408,904 reviews across 301,553 active users and 349,741 active items (99.9958% sparsity).
  * `data/books_50pct_users_meta.parquet` — Matched metadata records for all 349,741 surviving books.

## Execution Notes
* **Runtime:** ~120 minutes total (dominated by network download speeds; local parsing takes ~6 minutes).
* **Caching:** If the output `.parquet` files already exist in `data/`, the script skips execution automatically. Delete those files if you wish to regenerate the dataset from scratch.

In [6]:
import gzip
import json
import os
import time
import urllib.request
import zlib
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [7]:
# ---------------- Configuration ----------------
CATEGORY = "Books"
FRACTION = 0.5     # Keep 50% of total users
SEED = 42          # Seed for deterministic hashing
MIN_COUNT = 2      # Minimal interaction threshold (k-core) for recommender stability

# Stanford SNAP URLs
REVIEWS_URL = f"https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_{CATEGORY}_5.json.gz"
META_URL = f"https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_{CATEGORY}.json.gz"

OUT_DIR = Path("data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_REVIEWS_FILE = OUT_DIR / f"reviews_{CATEGORY}_5.json.gz"
TMP_REVIEWS_FILE = OUT_DIR / f"reviews_{CATEGORY}_5.json.gz.tmp"

LOCAL_META_FILE = OUT_DIR / f"meta_{CATEGORY}.json.gz"
TMP_META_FILE = OUT_DIR / f"meta_{CATEGORY}.json.gz.tmp"

OUT_REVIEWS = OUT_DIR / f"{CATEGORY.lower()}_50pct_users_reviews.parquet"
OUT_META = OUT_DIR / f"{CATEGORY.lower()}_50pct_users_meta.parquet"


def verify_gzip(file_path: Path) -> bool:
    """Verifies if a local gzip file is complete and uncorrupted."""
    if not file_path.exists() or file_path.stat().st_size == 0:
        return False
    try:
        with gzip.open(file_path, "rb") as f:
            f.seek(-4, os.SEEK_END)
        return True
    except Exception:
        return False


def download_file_safely(url: str, dest_path: Path, tmp_path: Path, max_retries: int = 5):
    """Downloads files safely in 1MB chunks with retries and temporary file validation."""
    if dest_path.exists():
        if verify_gzip(dest_path):
            print(f"  Valid file found on disk: {dest_path.name}")
            return
        else:
            print(f"  Removing corrupted local file: {dest_path.name}")
            dest_path.unlink()

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    chunk_size = 1024 * 1024  # 1 MB

    for attempt in range(1, max_retries + 1):
        try:
            print(f"  Downloading {dest_path.name} (Attempt {attempt}/{max_retries})...")
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as resp, open(tmp_path, "wb") as f:
                total_size = int(resp.headers.get("Content-Length", 0))
                with tqdm(total=total_size, unit="B", unit_scale=True, desc=dest_path.name) as pbar:
                    while True:
                        chunk = resp.read(chunk_size)
                        if not chunk:
                            break
                        f.write(chunk)
                        pbar.update(len(chunk))

            tmp_path.rename(dest_path)
            print("  Download complete and verified!")
            return
        except Exception as e:
            print(f"  Warning: Download interrupted ({e}).")
            if tmp_path.exists():
                tmp_path.unlink()
            if attempt < max_retries:
                time.sleep(5)
            else:
                raise RuntimeError(f"Failed to download {url} after {max_retries} attempts.")


def is_user_in_sample(user_id: str, fraction: float = 0.5, seed: int = 42) -> bool:
    """Deterministic user sampling via CRC32 hashing."""
    key = f"{user_id}_{seed}".encode("utf-8")
    return (zlib.crc32(key) % 10_000) < (fraction * 10_000)


def safe_parse_line(line: str) -> dict:
    """Parses JSON or Python dict format safely."""
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        return eval(line)  # Fallback for older SNAP string-formatted dicts


def parse_nested_details(details_obj: dict) -> dict:
    """Extracts author, page count, publisher, and language from the details dictionary."""
    if not isinstance(details_obj, dict):
        return {"author": None, "page_count": None, "publisher": None, "language": None}

    d = {str(k).lower().strip(): str(v) for k, v in details_obj.items()}
    raw_pages = d.get("page count") or d.get("print length") or d.get("paperback") or d.get("hardcover")

    return {
        "author": d.get("author") or d.get("by (author)"),
        "page_count": raw_pages,
        "publisher": d.get("publisher"),
        "language": d.get("language"),
    }


def build_recommender_dataset():
    if OUT_REVIEWS.exists() and OUT_META.exists():
        print("Outputs already exist. Delete files in 'data/' to re-run.")
        return

    # ----------------------------------------------------
    # Step 1: Download & Stream Local Reviews
    # ----------------------------------------------------
    print("Step 1: Preparing local reviews dataset...")
    download_file_safely(REVIEWS_URL, LOCAL_REVIEWS_FILE, TMP_REVIEWS_FILE)

    print(f"\nStep 2: Sampling {FRACTION * 100:.0f}% of users from local reviews...")
    review_records = []
    total_parsed_lines = 0

    with gzip.open(LOCAL_REVIEWS_FILE, "rt", encoding="utf-8") as f:
        for line in tqdm(f, desc="Parsing Reviews"):
            total_parsed_lines += 1
            data = safe_parse_line(line)
            user_id = data.get("reviewerID")

            if user_id and is_user_in_sample(user_id, fraction=FRACTION, seed=SEED):
                review_records.append({
                    "user_id": user_id,
                    "item_id": data.get("asin"),
                    "rating": float(data.get("overall", 0.0)),
                    "summary": data.get("summary", ""),
                    "review_text": data.get("reviewText", ""),
                    "helpful_votes": data.get("helpful"),
                    "timestamp": data.get("unixReviewTime"),
                })

    reviews_df = pd.DataFrame(review_records)
    del review_records

    print(
        f"\n  Raw Stream Summary:\n"
        f"  - Total reviews evaluated: {total_parsed_lines:,}\n"
        f"  - Sampled user reviews kept: {len(reviews_df):,}\n"
        f"  - Unique sampled users: {reviews_df['user_id'].nunique():,}\n"
        f"  - Unique items interacted with: {reviews_df['item_id'].nunique():,}"
    )

    # ----------------------------------------------------
    # Step 2: k-core Filtering
    # ----------------------------------------------------
    if MIN_COUNT > 1:
        print(f"\nStep 3: Applying {MIN_COUNT}-core iterative filter...")
        while True:
            n_start = len(reviews_df)

            # Filter users
            user_counts = reviews_df["user_id"].value_counts()
            reviews_df = reviews_df[reviews_df["user_id"].isin(user_counts[user_counts >= MIN_COUNT].index)]

            # Filter items
            item_counts = reviews_df["item_id"].value_counts()
            reviews_df = reviews_df[reviews_df["item_id"].isin(item_counts[item_counts >= MIN_COUNT].index)]

            if len(reviews_df) == n_start:
                break

        print(
            f"  Core Filtered Summary:\n"
            f"  - Retained reviews: {len(reviews_df):,}\n"
            f"  - Active users: {reviews_df['user_id'].nunique():,}\n"
            f"  - Active items: {reviews_df['item_id'].nunique():,}"
        )

    keep_items = set(reviews_df["item_id"].unique())

    # ----------------------------------------------------
    # Step 3: Download & Stream Local Metadata
    # ----------------------------------------------------
    print("\nStep 4: Preparing local metadata dataset...")
    download_file_safely(META_URL, LOCAL_META_FILE, TMP_META_FILE)

    print("\nStep 5: Extracting metadata for surviving items...")
    meta_records = []

    with gzip.open(LOCAL_META_FILE, "rt", encoding="utf-8") as f:
        for line in tqdm(f, desc="Parsing Metadata"):
            data = safe_parse_line(line)
            asin = data.get("asin") or data.get("parent_asin")

            if asin in keep_items:
                details_info = parse_nested_details(data.get("details", {}))
                features_list = data.get("features") or data.get("feature") or []
                features_text = " ".join(features_list) if isinstance(features_list, list) else str(features_list)

                meta_records.append({
                    "item_id": asin,
                    "title": data.get("title", ""),
                    "author": details_info["author"],
                    "page_count": details_info["page_count"],
                    "publisher": details_info["publisher"],
                    "language": details_info["language"],
                    "description": data.get("description", ""),
                    "features": features_text,
                    "price": data.get("price"),
                    "brand": data.get("brand", ""),
                    "categories": data.get("categories", []),
                    "sales_rank": data.get("salesRank"),
                    "related_items": data.get("related") or data.get("bought_together"),
                })

    meta_df = pd.DataFrame(meta_records).drop_duplicates("item_id")
    del meta_records

    print(f"  Matched metadata for {len(meta_df):,} out of {len(keep_items):,} items.")

    # ----------------------------------------------------
    # Step 4: Save Parquet Files
    # ----------------------------------------------------
    print("\nStep 6: Saving datasets to Parquet...")
    reviews_df.to_parquet(OUT_REVIEWS, index=False)
    meta_df.to_parquet(OUT_META, index=False)

    # ----------------------------------------------------
    # Summary Statistics
    # ----------------------------------------------------
    n_u = reviews_df["user_id"].nunique()
    n_i = reviews_df["item_id"].nunique()
    n_r = len(reviews_df)
    avg_text_length = reviews_df["review_text"].str.split().str.len().mean()
    sparsity = 100.0 * (1 - (n_r / (n_u * n_i)))

    print("\n" + "=" * 50)
    print("      NLP & RECOMMENDER DATASET PREPARATION COMPLETE")
    print("=" * 50)
    print(f" Saved Reviews File:  {OUT_REVIEWS}")
    print(f" Saved Metadata File: {OUT_META}")
    print(f"\n Dataset Summary:")
    print(f"  - Users (N):            {n_u:,}")
    print(f"  - Items (M):            {n_i:,}")
    print(f"  - Total Reviews:        {n_r:,}")
    print(f"  - Avg Words / Review:   {avg_text_length:.1f}")
    print(f"  - Matrix Sparsity:      {sparsity:.4f}%")
    print("=" * 50)


if __name__ == "__main__":
    build_recommender_dataset()

Step 1: Preparing local reviews dataset...


reviews_Books_5.json.gz: 100%|██████████| 3.22G/3.22G [1:11:24<00:00, 752kB/s] 


  Download complete and verified!

Step 2: Sampling 50% of users from local reviews...


Parsing Reviews: 8898041it [02:09, 68605.66it/s] 



  Raw Stream Summary:
  - Total reviews evaluated: 8,898,041
  - Sampled user reviews kept: 4,424,388
  - Unique sampled users: 301,554
  - Unique items interacted with: 365,224

Step 3: Applying 2-core iterative filter...
  Core Filtered Summary:
  - Retained reviews: 4,408,904
  - Active users: 301,553
  - Active items: 349,741

Step 4: Preparing local metadata dataset...


meta_Books.json.gz: 100%|██████████| 826M/826M [19:44<00:00, 697kB/s]  


  Download complete and verified!

Step 5: Extracting metadata for surviving items...


Parsing Metadata: 2370585it [03:37, 10880.15it/s]


  Matched metadata for 349,741 out of 349,741 items.

Step 6: Saving datasets to Parquet...

      NLP & RECOMMENDER DATASET PREPARATION COMPLETE
 Saved Reviews File:  data/books_50pct_users_reviews.parquet
 Saved Metadata File: data/books_50pct_users_meta.parquet

 Dataset Summary:
  - Users (N):            301,553
  - Items (M):            349,741
  - Total Reviews:        4,408,904
  - Avg Words / Review:   143.4
  - Matrix Sparsity:      99.9958%
